# Option-Chain Data Snapshot

This notebook pulls raw option-chain quotes for one ticker across multiple expiries and saves a frozen CSV snapshot. The raw snapshot is kept separate from later cleaning, filtering, implied-volatility calculations, and Greek calculations.

## Option-chain fields

An option chain is the listed set of option contracts for an underlying ticker. The chain is organized by expiry date, option type, and strike.

| Field | Meaning |
|---|---|
| `option_type` | Call or put |
| `expiration` | Contract expiry date |
| `strike` | Exercise price |
| `bid` | Price buyers are currently willing to pay |
| `ask` | Price sellers are currently willing to accept |
| `mid_price` | Average of bid and ask |
| `lastPrice` | Most recent traded price |
| `volume` | Contracts traded during the current session |
| `openInterest` | Contracts currently open |
| `impliedVolatility` | Vendor-provided implied volatility estimate |

For this project, bid and ask prices are central because they show that the market price is a range, not one exact number.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

In [ ]:
project_root = Path.cwd()

if not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

In [ ]:
from src.data_loader import (
    build_sample_option_chain,
    fetch_option_chains,
    load_option_chain_snapshot,
    save_option_chain_snapshot,
)

## Pull live option-chain data

The default ticker is `SPY`, but the same workflow can be run for `QQQ` or `AAPL`. The nearest available expiries are used unless specific expiry dates are supplied.

In [ ]:
ticker_symbol = "SPY"
max_expiries = 4
raw_output_dir = project_root / "data" / "raw"

try:
    raw_options = fetch_option_chains(ticker_symbol=ticker_symbol, max_expiries=max_expiries)
    data_status = "live"
except Exception as exc:
    raw_options = build_sample_option_chain(ticker_symbol=ticker_symbol)
    data_status = f"sample fallback: {exc}"

snapshot_path = save_option_chain_snapshot(
    raw_options,
    output_dir=raw_output_dir,
    ticker_symbol=ticker_symbol,
)

snapshot_path

## Raw option-chain table

The table below is the frozen snapshot that will be used as the starting point for later quote-quality and implied-volatility work.

In [ ]:
raw_options.head(10)

In [ ]:
summary = pd.DataFrame(
    {
        "metric": [
            "data_status",
            "rows",
            "ticker",
            "expiries",
            "calls",
            "puts",
            "snapshot_path",
        ],
        "value": [
            data_status,
            len(raw_options),
            ticker_symbol,
            raw_options["expiration"].nunique() if "expiration" in raw_options.columns else None,
            int((raw_options["option_type"] == "call").sum()) if "option_type" in raw_options.columns else None,
            int((raw_options["option_type"] == "put").sum()) if "option_type" in raw_options.columns else None,
            str(snapshot_path),
        ],
    }
)

summary

## Column reference

The raw vendor columns should be preserved before any filtering is applied. Later modules can remove stale quotes, zero bid/ask quotes, extremely wide spreads, and contracts with missing liquidity fields.

In [ ]:
column_reference = pd.DataFrame(
    {
        "column": raw_options.columns,
        "non_null_count": [raw_options[column].notna().sum() for column in raw_options.columns],
        "dtype": [str(raw_options[column].dtype) for column in raw_options.columns],
    }
)

column_reference

## Bid, ask, and mid prices

The bid-ask spread is a direct quote-quality feature. A wider spread means the option's executable market price is less precise.

In [ ]:
quote_columns = [
    column
    for column in [
        "ticker",
        "expiration",
        "option_type",
        "contractSymbol",
        "strike",
        "bid",
        "ask",
        "mid_price",
        "lastPrice",
        "volume",
        "openInterest",
        "impliedVolatility",
    ]
    if column in raw_options.columns
]

raw_options[quote_columns].head(15)

## Reload the frozen snapshot

Reloading the CSV confirms that later analysis can start from the saved file instead of pulling a fresh chain. This matters because option quotes change throughout the trading day.

In [ ]:
reloaded_snapshot = load_option_chain_snapshot(snapshot_path)
reloaded_snapshot.head(10)

## Snapshot notes

Public option-chain data can be noisy. Quotes may be stale, bid or ask values may be missing, and vendor-provided implied volatility may not match the project's own calculation. The raw snapshot is therefore treated as input data, not as cleaned analysis output.

Later analysis should calculate bid IV, mid IV, and ask IV directly from the saved quote prices instead of relying blindly on the vendor implied-volatility column.